Install dependencies

In [ ]:
import torch

print("CUDA Available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
!pip install -q ultralytics roboflow

Download dataset (Roboflow)

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="zkWepRCcCww5IjHVvkJw")
project = rf.workspace("ttsmokefire").project("smoke-fire-s2oxt")
version = project.version(7)
dataset = version.download("yolov8")


Verify dataset

In [ ]:
import os

print(dataset.location)
print(os.listdir(dataset.location))

In [ ]:
with open(f"{dataset.location}/data.yaml","r") as f:
    print(f.read())

Count images and labels

In [ ]:
import os

print("Training   :", len(os.listdir(f"{dataset.location}/train/images")))
print("Validation :", len(os.listdir(f"{dataset.location}/valid/images")))
print("Testing    :", len(os.listdir(f"{dataset.location}/test/images")))

In [ ]:
print("Train labels :", len(os.listdir(f"{dataset.location}/train/labels")))
print("Valid labels :", len(os.listdir(f"{dataset.location}/valid/labels")))
print("Test labels  :", len(os.listdir(f"{dataset.location}/test/labels")))

Visualize annotations

In [ ]:
from ultralytics.data.utils import visualize_image_annotations
import glob
import random
import yaml # Import yaml library

# Load the data.yaml file to get label names
with open(f"{dataset.location}/data.yaml", "r") as f:
    data_yaml = yaml.safe_load(f)
    label_map = data_yaml['names']

images = glob.glob(f"{dataset.location}/train/images/*.jpg")

for img in random.sample(images, 5):
    label = img.replace("/images/", "/labels/").replace(".jpg", ".txt")
    visualize_image_annotations(img, label, label_map)


Count Fire vs Smoke objects

In [ ]:
import glob

fire = 0
smoke = 0

for txt in glob.glob(f"{dataset.location}/train/labels/*.txt"):
    with open(txt) as f:
        for line in f:
            cls = int(line.split()[0])
            if cls == 0:
                fire += 1
            elif cls == 1:
                smoke += 1

print("Fire objects :", fire)
print("Smoke objects:", smoke)

Train YOLOv8m

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8m.pt")

results = model.train(
    data=f"{dataset.location}/data.yaml",

    epochs=100,
    imgsz=640,
    batch=16,

    optimizer="AdamW",
    lr0=0.001,
    weight_decay=0.0005,

    patience=25,

    device=0,
    workers=2,

    pretrained=True,
    amp=True,
    cache=True,

    cos_lr=True,

    project="FireSmokeProject",
    name="YOLOv8m_FireSmoke",

    plots=True
)

Validate best model

In [ ]:
from ultralytics import YOLO

best = YOLO("/content/runs/detect/FireSmokeProject/YOLOv8m_FireSmoke/weights/best.pt")

metrics = best.val(data=f"{dataset.location}/data.yaml")

print("Precision :", metrics.box.mp)
print("Recall    :", metrics.box.mr)
print("mAP50     :", metrics.box.map50)
print("mAP50-95  :", metrics.box.map)

In [ ]:
from google.colab import files
import shutil

source = "/content/runs/detect/FireSmokeProject/YOLOv8m_FireSmoke/weights/best.pt"
destination = "/content/optimized150.pt"

shutil.copy(source, destination)
files.download(destination)